# 6.1 Observability & Tracing (LangSmith)

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

**Prerequisites:** Module 4 (graphs, streaming, interrupts), Module 5 (a multi-agent run worth tracing). You need a free LangSmith account and an API key for the parts that send data; every code sample runs without a key too, and says what happens in that case.

## Learning objectives

- Explain why agents need tracing more than chains do, in terms of run length, branching, and money.
- Enable tracing with the current environment variables, name projects deliberately, and pick the EU endpoint when data residency matters.
- Read a run tree: nested runs, inputs and outputs, latency, token counts, and how to find the one step that made a run expensive.
- Attach metadata and tags to runs with the config dict, the `@traceable` decorator, and the `trace` context manager.
- Describe what tracing costs, what the retention tiers mean, and what the self-hosted option buys you.
- Treat traces as production data: prompts and outputs inside, redaction before sharing, a retention policy from day one.

## Why agents need tracing more than chains

A chain is a fixed sequence. When it misbehaves, you read five steps and find the bad one. An agent run is different in three ways, and each one makes tracing more valuable.

First, **length**. A research agent (your 5.5 project) can run dozens of model calls, tool calls, and retrievals. Nobody reads that from logs. A run tree shows it as nested, collapsible steps, and you expand only the branch that smells wrong.

Second, **branching**. Parallel researchers, a critic loop that may or may not fire, a conditional edge that took the path you did not expect. Logs flatten this into a sequence and lie about the shape. The run tree preserves it.

Third, **money**. This is the one that gets budget approval for observability tooling. An agent run that calls a model forty times has a cost, and when someone asks "what did it do with my money", the answer is the trace: which steps ran, how many tokens each consumed, and which step was the expensive one. LangSmith's dashboard aggregates token usage and estimated cost per run and per project, which turns "the agent is expensive" into "the critic loop is 60% of the cost".

Tracing is also how you debug the failure modes from earlier modules without guessing: the wrong document retrieved (3.x), the tool that returned garbage (2.x), the interrupt payload that was never read (4.4). The trace shows the actual inputs and outputs of each step, so you stop blaming the model for what was actually a retrieval bug.

## Enabling tracing: the current setup

Tracing is controlled by environment variables, read when your process starts. Set these in your shell or `.env` file, not halfway through the script. (The SDK caches the environment lookup, so setting `LANGSMITH_TRACING` after the first traced call in a process may not take effect. Verified behaviour on langsmith 0.14.11: set it before the process starts.)

In [ ]:
!export LANGSMITH_TRACING=true
!export LANGSMITH_API_KEY="lsv2_pt_..."        # from your LangSmith settings page
!export LANGSMITH_PROJECT="ai-agents-course"   # optional; defaults to "default"
# export LANGSMITH_ENDPOINT="https://eu.api.smith.langchain.com"  # EU region only

The variable names matter because the old ones are still floating around in tutorials. The current canonical name is `LANGSMITH_TRACING`. The older `LANGSMITH_TRACING_V2` (and the older-still `LANGCHAIN_TRACING_V2` / `LANGCHAIN_TRACING`) are still honoured by the SDK as fallbacks, but write the current name in new code. `LANGSMITH_PROJECT` groups runs; use one project per application or per environment (`myapp-dev`, `myapp-prod`), not one project for everything. `LANGSMITH_ENDPOINT` defaults to the US endpoint; set it to the EU endpoint if you signed up in the EU region or need EU data residency. The EU angle is not cosmetic: it decides which continent your prompts are stored on, and 6.5 returns to why that choice matters under GDPR.

With `LANGSMITH_TRACING=true`, LangChain and LangGraph auto-trace. No decorators, no wrappers, no client code. The callback system attaches a tracer to every invocation. That is the whole pitch, and it works for any model provider, because the tracing comes from LangChain's callback layer, not from the model SDK.

What if the key is missing or the network is down? Verified on 0.14.11: the call still works. The SDK logs a warning (`Failed to ... ingest runs`) and your application continues. Tracing failures are non-fatal by design. This is worth knowing twice: once so you are not afraid to leave tracing on in production, and once so you do not mistake "no traces in the dashboard" for "tracing is off". Check the key, the endpoint, and the project name before you blame the code.

## Your first traced run

This LCEL pipeline runs with no API key and no LLM. With tracing on and a key set, it appears in your project as a run tree. Without a key, it returns the same answer and logs a warning.

In [ ]:
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "course-6.1-demo"
# LANGSMITH_API_KEY is read from your environment; without it the SDK
# warns and the pipeline still returns its answer.

from langchain_core.runnables import RunnableLambda

def clean(text: str) -> str:
    return text.strip().lower()

def count_words(text: str) -> int:
    return len(text.split())

pipeline = (
    RunnableLambda(clean, name="clean_text")
    | RunnableLambda(count_words, name="count_words")
)

result = pipeline.invoke(
    "  Agents need observability  ",
    config={
        "run_name": "word-count-demo",
        "tags": ["module-6", "lesson-6.1", "demo"],
        "metadata": {"course": "ai-agents-course", "env": "dev"},
    },
)
print(result)   # 3

Open the `course-6.1-demo` project in the LangSmith UI and find the run named `word-count-demo`. This is what a run tree shows you, top to bottom:

- **The root run**, named `word-count-demo`, of run type `chain`. Its inputs are the raw input (`"  Agents need observability  "`); its outputs are `3`. Total latency for the whole pipeline.
- **Child runs**, one per step: `clean_text` and `count_words`, each with their own inputs, outputs, and latency. The tree nests exactly the way your pipeline composes.
- **Per-run timing**, so you can see which step dominated. On this toy it does not matter; on a 5.5 research run, this column is where you discover the critic loop ate 60% of the budget.
- **Token counts and cost**, on runs that call a model. Each LLM child run records prompt tokens, completion tokens, and the model name, and the dashboard rolls them into cost estimates per run and per project.

Two fields in the `config` dict deserve a closer look, because they are how you find runs later. `tags` are labels for filtering: `module-6`, `env:prod`, `customer:acme`. `metadata` is key-value context: the course name, the git commit, the model version. Tags answer "show me all runs for this feature"; metadata answers "which model version produced this run". Set both as a habit, on every traced entrypoint, from the first prototype. Retrofitting tags onto a thousand untagged runs is nobody's idea of a good afternoon.

## Tracing your own code: `@traceable` and friends

Auto-tracing covers LangChain and LangGraph calls. Your own functions, preprocessing, business logic, the quote-match evaluator from 5.5, need the SDK's explicit tools. All of these were verified against langsmith 0.14.11.

The `@traceable` decorator turns any function into a traced run. Nested decorated calls nest in the tree automatically.

In [ ]:
import langsmith as ls

@ls.traceable(
    name="evaluate-faithfulness",
    run_type="chain",
    tags=["eval", "module-5-project"],
    metadata={"evaluator": "quote-match", "version": "1.0"},
)
def evaluate_faithfulness(draft: str, findings: list) -> dict:
    # ... the quote-match check from lesson 5.5 ...
    return {"score": 0.95, "rows": []}

Tags and metadata can also be added at call time, which is how you tag one invocation differently from the function's defaults:

In [ ]:
evaluate_faithfulness(
    draft,
    findings,
    langsmith_extra={"tags": ["nightly-eval"], "metadata": {"dataset": "briefs-v3"}},
)

For code that is not a single function, `ls.trace` opens a span as a context manager, and `ls.tracing_context` sets default metadata for everything inside the block without creating a span of its own:

In [ ]:
with ls.tracing_context(metadata={"request_id": "req-42", "env": "prod"}):
    result = graph.invoke({"question": brief}, config)

with ls.trace(name="publish_report", run_type="chain", inputs={"brief": brief}) as rt:
    write_report(result["draft"])

And when the metadata depends on something you only learn mid-run, `get_current_run_tree()` gives you the current span to annotate:

In [ ]:
rt = ls.get_current_run_tree()
if rt is not None:
    rt.metadata["revisions_used"] = state["revisions"]
    rt.tags.append("critic-loop-fired")

A note on honesty: `@traceable` on a plain function records inputs and outputs. That includes whatever you pass in. If your function handles user data, the trace now contains user data. This is the privacy section below, arriving early, because it is the single most common tracing mistake.

## Filtering and inspecting runs

The UI is where traces pay off. The workflows that matter daily:

- **Find the bad run.** Filter by tags (`env:prod`), by metadata (`model_version`), by time window, by error status. When a user reports "the agent gave a wrong answer at 14:05", you filter to that window and open the run.
- **Read it top-down.** Start at the root run's outputs, then expand children only where the output looks wrong. The discipline is the same as debugging with a stack trace: start at the symptom, walk toward the cause.
- **Compare inputs and outputs at each step.** The retrieval step's output is where RAG bugs live: the wrong chunk retrieved, the right chunk ranked fifth. Most "model" bugs are retrieval bugs wearing a costume.
- **Share the run.** LangSmith generates shareable links to individual runs, which is how you show a colleague the exact failure instead of describing it. Before you share, read the privacy section below.
- **Attach feedback.** You can score runs in the UI or via the API. Feedback is how traces become eval data (lesson 6.2 builds on this). One billing behaviour to know: a trace that receives feedback is automatically promoted to extended retention, which costs more. Do not run bulk feedback jobs without knowing that.

## What tracing costs

The pricing shape, as of late 2026 (check the current pricing page before you budget; the per-trace rates moved more than once during 2026):

- **Developer (free):** 5,000 base traces per month, one seat, 14-day retention. Enough for this course and for prototyping.
- **Plus:** $39 per seat per month, 10,000 base traces included, then pay per thousand traces.
- **Enterprise:** custom pricing, and the only tier that permits self-hosting or a hybrid data plane.

Two retention tiers: **base** traces (14 days) and **extended** traces (400 days, higher per-trace rate). Remember the feedback rule above: feedback promotes a trace to extended retention automatically. An eval pipeline that scores every production run is also a pipeline that upgrades every production run's retention. Budget for it.

The honest cost model for agents: one agent run is one trace, but the trace contains dozens of nested runs, and you are billed per trace, not per nested run. A research team run that fans out to five researchers costs one trace. What costs real money is volume: thousands of runs a day, each with full prompt and completion payloads. Two levers control it. First, sample: trace every run in dev, a fraction in prod. Second, keep payloads small: a trace that stores a 2 MB document on every run is a storage bill wearing an observability costume. The SDK supports hiding inputs, outputs, or metadata per client if you need it.

## The self-hosted option

LangSmith can run inside your own infrastructure. The shape, at a high level: LangChain ships container images and a Helm chart; you provide the Kubernetes cluster, PostgreSQL, Redis, and object storage; you hold a license key from the sales team. There is also a hybrid topology where your agent workloads run in your infrastructure and send traces to LangSmith Cloud.

Why it exists: data residency, regulatory constraints, custom networking, air-gapped requirements. When it is worth it: when your traces contain data that cannot leave your network, or when trace volume makes SaaS pricing absurd. When it is not: when you are a team of three and the SaaS free tier covers you. Self-hosting trades a vendor bill for an operations burden: upgrades, scaling, and backups become your job. This lesson notes that the option exists and what it costs in operations terms. The install walkthrough belongs to infrastructure docs, not to this course.

## Privacy: traces are production data

Everything in this section follows from one fact: **a trace contains your prompts and the model's outputs.** If the prompt contains a user's message, the trace contains the user's message. If the RAG context contains a private document, the trace contains the private document.

Treat traces accordingly:

- **Redact before sharing.** Share links are convenient and permanent-feeling. Strip or mask personal data before a trace leaves your team.
- **Set a retention policy.** The default retention is a choice you made by not choosing. Decide how long traces live, especially any that touch personal data, and configure it rather than discovering it during an audit.
- **Choose the region deliberately.** The EU endpoint exists so EU personal data can stay in the EU. Picking it is a one-line config change with real legal weight.

The EU angle, stated plainly: traces containing personal data are processing of personal data under GDPR. That means a lawful basis, data minimisation (do not trace fields you do not need), retention limits, and honouring deletion requests, which is awkward when the data lives in a third-party dashboard with 400-day retention. Your 6.5 lesson covers the deployer's checklist. For now, the rule is: know what is in your traces before your first production run, not after your first incident.

## What can go wrong

- **Tracing never turned on.** `LANGSMITH_TRACING` set after the process started, or set in the wrong shell, or the `.env` file never loaded. The app works fine and the dashboard is empty, which looks exactly like "tracing is broken". Verify with one traced call and check the project, before you debug anything else.
- **Everything in one project.** The default project becomes a swamp: dev runs, prod runs, eval runs, all mixed. Name projects per app and environment from the start. Renaming later does not untangle history.
- **Personal data in traces, discovered late.** The `@traceable` function that takes a user profile dict now stores user profiles in the dashboard. Audit what your traced functions receive before production, not after a subject access request.
- **Feedback promoting retention silently.** An eval job scores ten thousand runs and every one of them jumps to extended retention. Read the billing docs on auto-extended retention before wiring feedback into a pipeline.
- **Enormous trace payloads.** Logging full documents, images, or embeddings on every run. Traces are for debugging, not for archiving your corpus. Keep the corpus in storage; keep references in the trace.
- **Tracing the wrong layer.** Auto-tracing a graph whose nodes each make five internal model calls gives you a tree with hundreds of leaves. If the internal calls are noise, they are still billed as trace content and still slow down the UI. Trace the boundaries that matter: the agent's decisions, the tool calls, the final answer.
- **Assuming the EU endpoint is automatic.** It is not. The default endpoint is US. If your users are in the EU and your traces carry personal data, setting `LANGSMITH_ENDPOINT` to the EU endpoint is a deliberate, documented decision. Make it once, on purpose.

## Check your understanding

**1. Why do agents need tracing more than fixed chains do? Give the three reasons from this lesson.**
Length: agent runs span dozens of steps, unreadable in flat logs but navigable as a nested tree. Branching: parallel workers and conditional loops have a shape that logs flatten and misrepresent. Money: multi-step runs have real per-run cost, and the trace attributes tokens and cost to individual steps, answering "what did it do with my money" precisely.

**2. You set `LANGSMITH_TRACING=true` in your shell, run your script, and the dashboard is empty, but the script worked. Name two distinct causes and how you would tell them apart.**
One: the variable was set after the process's first traced call, or in a different shell than the one running the script; the SDK caches the environment lookup. Check by printing the variable inside the running process. Two: the key or project is wrong, for example traces landing in the `default` project while you stare at another, or the SDK warning about a failed ingest scrolling past in the logs. Check the project's run list and the process's stderr for the ingest warning. A third possibility is that tracing was never on for that process at all, which the in-process print also reveals.

**3. What is the difference between tags and metadata, and give one example of each you would set on a production agent run.**
Tags are labels for filtering and grouping: `env:prod`, `customer:acme`, `feature:research-team`. Metadata is key-value context attached to the run: `model_version`, `git_commit`, `prompt_template_version`. Tags answer "show me every run for this slice"; metadata answers "under what conditions did this specific run execute". Both are set via the `config` dict on invoke, and both are searchable in the UI.

**4. Your nightly eval job adds feedback scores to every production trace. What billing consequence should you expect, and why?**
Each trace that receives feedback is automatically promoted from base to extended retention (400 days), which is billed at the higher per-trace rate. A job that scores every production run therefore upgrades every production run's retention. Budget for the extended rate on the full production volume, or score a sample, before wiring feedback into the pipeline.

**5. A teammate proposes self-hosting LangSmith to "save money" for a three-person team on the free tier. What is wrong with this reasoning?**
The free Developer tier already covers their volume at zero cost, so there is no bill to save. Self-hosting trades the SaaS bill for an operations burden: Kubernetes, PostgreSQL, Redis, upgrades, scaling, and backups become the team's job. The self-hosted option pays off when traces cannot leave your network or when trace volume makes SaaS pricing absurd, neither of which describes a three-person team on the free tier.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.